In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/distributed_grpo_trainer.py
/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/plan_overview.md
/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/inference_check.py
/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/data_pipeline.py
/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/medqa_dataset/phrases_no_exclude_train.jsonl
/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/medqa_dataset/phrases_no_exclude_test.jsonl
/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/medqa_dataset/.cache/huggingface/.gitignore
/kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/medqa_dataset/.cache/huggingface/download/

# Project Overview: Gemma-Sync 🚀

**Sub-title:** Distributed Uncertainty-Aware Clinical Reasoning via Gemma 4 E2B

---

## 1. Executive Summary

Gemma-Sync is a clinical AI project designed to fine-tune **Gemma 4 E2B** using **Group Relative Policy Optimization (GRPO)** and **Reinforcement Learning from Verifiable Rewards (RLVR)**. The project bridges the gap between high-performance cloud training and local-first reliable deployment, utilizing the **SOFA (Sequential Organ Failure Assessment)** score as a verifiable logic constraint to mitigate hallucinations.

## 2. Hardware Architecture & Prize Track Alignment

This project is strategically architected to qualify for multiple **Special Technology Tracks**:

| Stage                  | Hardware         | Tech Stack                   | 
| ---------------------- | ---------------- | ---------------------------- | 
| **Prototyping**  | MacBook M2 (MPS) | PyTorch, Transformers        | 
| **Training**     | Kaggle TPU v5e-8 | JAX / PyTorch XLA            | 
| **Optimization** | Kaggle 2x T4 GPU | **Unsloth**            | 
| **Deployment**   | MacBook M2       | **Ollama / llama.cpp** |

### 2.5 Distribution Architecture (Distributed Learning Focus)

To handle the computational demands of RLVR and GRPO, this project utilizes a **Synchronous Data Parallelism** approach across a multi-core TPU topology:

* **8-Way Data Parallelism (TPU v5e-8):** The training data is partitioned into 8 distinct shards. Each of the 8 TPU cores maintains a replica of the Gemma 4 E2B weights and processes its respective data shard simultaneously.
* **Gradient Synchronization (All-Reduce):** After each forward and backward pass, gradients are synchronized across all 8 cores using an **All-Reduce** collective communication primitive. This ensures that the global model updates are consistent across the entire distributed cluster.
* **GRPO Group Sharding:** The group generation step ($G=4$) is distributed across the cores. For every prompt, the 8 cores cooperate to generate the candidate response pool, allowing the **Relative Advantage** calculation to be computed in parallel rather than sequentially.
* **XLA Graph Compilation:** The entire training loop is compiled into an optimized execution graph via  **OpenXLA** . This minimizes the communication overhead between the CPU host and the TPU devices, maximizing the FLOPS utilization for clinical reasoning tasks.

### 2.6 Distributed Efficiency Metrics

* **Total Group Size (G):** 16 candidates (2 per core x 8 cores).
* **Synchronization Method:** All-Gather for reward aggregation across the TPU backplane.
* **Memory Headroom:** Leveraging the small footprint of E2B (~4GB) to maintain high-speed parallel generation within the 16GB per-core limit.

## 3. Core Methodology: Calibrated Reasoning

The model is trained to be "metacognitively aware" by integrating psychological principles of calibration into the AI's reasoning loop:

* **Verifiable SOFA Logic:** A hard-coded verifier checks the extraction and calculation of 6 organ systems (Respiratory, Coagulation, Liver, Cardiovascular, CNS, Renal).
* **Calibrated Abstention:** Using RLVR to reward the model for correctly flagging cases with insufficient clinical data ("I don't know") rather than guessing.
* **Intelligent Routing (Cactus-style):** The local M2 instance routes complex cases to high-parameter models only when local uncertainty thresholds are exceeded.

## 4. Technical Constraints

* **Base Model:** Gemma 4 E2B.
* **PEFT:** LoRA (Rank 32, Alpha 64).
* **Distributed Training:** 8-core TPU Parallelism (128GB VRAM).
* **Inference Format:** GGUF (4-bit/8-bit Quantization) for local performance.

---

`<small>`*Evolved from "Calibrated Abstention in Clinical LLMs" (Zenodo: 10.5281/zenodo.19599245).*`</small>`

In [2]:
#!rm -rf /kaggle/working/*

In [3]:
import os

# Copy all scripts to the working folder (T4)
!cp -r /kaggle/input/datasets/narendrabayutama/t4-version/* /kaggle/working/

# Copy all scripts to the working folder (TPU)
#!cp -r /kaggle/input/datasets/narendrabayutama/emergency-enabled-clinical-reasoning-via-grpo-and-rl/* /kaggle/working/

os.makedirs("/kaggle/working/outputs", exist_ok=True)

In [4]:
# Instalasi khusus untuk optimasi GPU NVIDIA (T4)
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U transformers peft accelerate trl datasets bitsandbytes

# Update library untuk TPU
#!pip install -U transformers peft accelerate trl datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-rms55tjf/unsloth_8a794084d5b8442bbaf60c2c001b3ee9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-rms55tjf/unsloth_8a794084d5b8442bbaf60c2c001b3ee9
  Resolved https://github.com/unslothai/unsloth.git to commit efed5c37394a144349cd9b1ea525e132e04584e5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.6 MB/s eta 0:00:00


In [5]:
#import json
#from pathlib import Path

# Ganti dengan path checkpoint kamu
#config_path = Path("/kaggle/input/models/narendrabayutama/medreason-gemma-4-e2b-it/transformers/default/1/adapter_config.json")

#if config_path.exists():
    #with open(config_path) as f:
        #config = json.load(f)
   # print(f"✓ Checkpoint Terdeteksi!")
   # print(f"LoRA Rank (r): {config.get('r')}")
   # print(f"LoRA Alpha: {config.get('lora_alpha')}")
   # print(f"Target Modules: {config.get('target_modules')}")
#else:
   # print("✗ File adapter_config.json tidak ditemukan!")

In [6]:
# Run the Training Script
#!python distributed_grpo_trainer.py \
 #   --batch-size 1 \
 #   --grad-accum 1 \
 #   --num-generations 4 \
 #   #--max-prompt-length 1024 \
 #   --max-completion-length 1024 \
 #   --max-steps 202 \
 #   --resume-from-checkpoint "/kaggle/input/models/narendrabayutama/medreason-gemma-4-e2b-it-cp300/transformers/default/1/outputs/gemma4-e2b-grpo-sofa-t4/checkpoint-100"

In [7]:
#!rm -rf /kaggle/working/outputs/*

In [8]:
## Print the latest results
#import json
#import matplotlib.pyplot as plt

## Load log history
#with open('/kaggle/working/outputs/gemma4-e2b-grpo-sofa-t4/checkpoint-200/trainer_state.json', 'r') as f:
  #  data = json.load(f)

## Ambil semua history reward
#rewards = [log['reward'] for log in data['log_history'] if 'reward' in log]

# Plot sederhana
#plt.plot(rewards)
#plt.title("Reward Progress Gemma-Sync")
#plt.show()

In [9]:
# Run the Inference Stage
#!python evaluation_final.py

In [10]:
# Run the Base Evaluation Stage
!python evaluation_base.py

2026-04-27 02:11:57 | INFO     | numexpr.utils | NumExpr defaulting to 4 threads.
2026-04-27 02:11:58 | INFO     | datasets | TensorFlow version 2.19.0 available.
2026-04-27 02:11:58 | INFO     | datasets | JAX version 0.7.2 available.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2026-04-27 02:12:34 | INFO     | base_eval | Loaded 11451 total raw records from medqa_dataset
2026-04-27 02:12:34 | INFO     | base_eval | ============================================================
2026-04-27 02:12:34 | INFO     | base_eval | AUDIT PHASE: Reproducing training split to build blacklist
2026-04-27 02:12:34 | INFO     | base_eval | ============================================================
2026-04-27 02:12:34 | INFO     | base_eval |   Combined dataset size: 11451
[Audit] Formatting for split reproduction (num_proc=2): 100%|█| 11451/11451 [00:
2026-04-27 02:12:37 | INFO     | base_eval |   StratifiedS